### Config Notebook

This notebook prepares shared configuration for the rest of the pipeline. It does not process any data itself — it only defines variables that other notebooks reuse.

* We read an `env` widget (`dev` or `prod`) to know which environment we are running in. If the widget is not set (for example, on a manual run), it defaults to `dev`.
* We load `pipeline_config.yml` and pick the section that matches the current environment.
* From that config, we build the full three-level table names (`catalog.schema.table`) for the bronze source table and the silver target table, so no other notebook has to hard-code them.
* We also store `business_key` (`video_id`) — this is the column that uniquely identifies a video across the whole pipeline. It is used later for deduplication and for every MERGE condition.

In [0]:
import yaml
import os

try:
    env = dbutils.widgets.get("env")
except Exception:
    env = "dev"

config_path = os.path.join(os.getcwd(), "pipeline_config.yml")
with open(config_path, "r") as file:
    full_config = yaml.safe_load(file)

config = full_config[env]

catalog = config["catalog"]
bronze_schema = config["bronze_schema"]
silver_schema = config["silver_schema"]
bronze_volumes_path = config["bronze_volumes_path"]

source_bronze_table = f"{catalog}.{bronze_schema}.{config['youtube_pipeline']['source_table']}"
target_silver_table = f"{catalog}.{silver_schema}.{config['youtube_pipeline']['target_table']}"
business_key = config["youtube_pipeline"]["business_key"]